# Radial Intensity Profile — Cellpose-SAM Version

Segments nuclei with **Cellpose-SAM (v4)** instead of StarDist. Cellpose handles densely packed and irregularly shaped nuclei in stem cell colonies noticeably better.

Computes per-channel radial intensity profiles from the center of the nuclear mask outward.

**Assumes 4D TIFFs (Z, C, Y, X) with 3 channels in this exact order:**
- C0 = ZO-1
- C1 = DAPI
- C2 = SMAD2

If your channel order differs, change `CHANNEL_NAMES` and `DAPI_IDX` in the parameters cell.


## Imports and parameters

In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt

from scipy.ndimage import center_of_mass
from skimage.measure import find_contours
from skimage import exposure
from cellpose import models
from pptx import Presentation
from pptx.util import Inches


In [ ]:
# === IMAGING PARAMETERS (change for your microscope) ===
SCALE_UM_PER_PX = 0.5             # microscope pixel size in µm
CHANNEL_NAMES = ['ZO-1', 'DAPI', 'SMAD2']
DAPI_IDX = 1                      # index of DAPI channel (0-based)

# === ANALYSIS PARAMETERS ===
MAX_RADIUS_UM = 250               # cut profile off at this radius (µm from center)
PLOT_MAX_UM = 300                 # x-axis extent for plots
NORMALIZE_PROFILES = True         # if True, each channel scaled to its own max=1

# === CELLPOSE PARAMETERS ===
# Diameter of a typical nucleus in PIXELS. For stem cell nuclei ~15-20 µm at 0.5 µm/px
# → ~30-40 px. Set to None for auto-estimate (slower, sometimes worse for dense colonies).
CELLPOSE_DIAMETER_PX = 35

# Tune if segmentation looks wrong:
#   Missing nuclei? Lower cellprob_threshold (-1) or flow_threshold (0.3)
#   Extra false nuclei? Raise cellprob_threshold (+1) or raise min_size
#   Nuclei merged?  Lower flow_threshold (0.3)
#   Nuclei split?   Raise flow_threshold (0.5)
CELLPOSE_FLOW_THRESHOLD = 0.4
CELLPOSE_CELLPROB_THRESHOLD = 0.0
CELLPOSE_MIN_SIZE = 15

# === PLOT COLORS (one per channel, matches CHANNEL_NAMES order) ===
CHANNEL_COLORS = ['green', 'blue', 'red']
CHANNEL_CMAPS  = ['plasma', 'inferno', 'magma']


In [ ]:
# Load Cellpose model ONCE (heavy download on first run)
cellpose_model = models.CellposeModel(gpu=False, pretrained_model='cpsam')


## Helpers

In [ ]:
def save_figure(fig, path, show=True):
    """Save at 300 dpi, optionally show, always close."""
    fig.savefig(path, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close(fig)


def clean_label(name):
    """Extract a readable label from filename. Adjust regex for your naming convention."""
    base = os.path.basename(name).replace('.tif', '')
    match = re.search(r'_(.*?)_488', base)
    return match.group(1).replace('_', ' ') if match else base


## Preprocessing and segmentation

In [ ]:
def load_and_preprocess(path):
    """Load 4D TIFF (Z, C, Y, X), max-project Z, return as (C, Y, X) float32."""
    img = tifffile.imread(path)
    if img.ndim != 4:
        raise ValueError(f"Expected 4D image (Z, C, Y, X), got shape {img.shape}")
    return img.max(axis=0).astype(np.float32)


def enhance_dapi(dapi):
    """
    Contrast-enhance the DAPI channel to help segmentation.
    Uses CLAHE (local histogram equalization) — handles uneven illumination across
    the field of view better than a single global percentile stretch.
    """
    dapi_norm = dapi / dapi.max() if dapi.max() > 0 else dapi
    return exposure.equalize_adapthist(dapi_norm.astype(np.float32), clip_limit=0.03)


def cellpose_mask(dapi_enhanced):
    """
    Segment nuclei with Cellpose-SAM.
    Returns (labels, binary_mask) where labels are integer IDs and
    binary_mask is uint8 (1 inside nuclei, 0 elsewhere).
    """
    masks, _flows, _styles = cellpose_model.eval(
        dapi_enhanced,
        diameter=CELLPOSE_DIAMETER_PX,
        flow_threshold=CELLPOSE_FLOW_THRESHOLD,
        cellprob_threshold=CELLPOSE_CELLPROB_THRESHOLD,
        min_size=CELLPOSE_MIN_SIZE,
        normalize=True,
    )
    return masks, (masks > 0).astype(np.uint8)


def compute_center_float(mask):
    """
    Center of mass of the nuclear mask.
    Returned as (x, y) floats — sub-pixel precision, no int truncation.
    """
    cy, cx = center_of_mass(mask)
    return (cx, cy)


## Radial profile

In [ ]:
def radial_profile_masked(data, center, mask):
    """
    Per-radius mean intensity from `data`, using ONLY pixels inside `mask`.
    Fast implementation using np.bincount (avoids the per-radius Python loop).
    Returns (radii_um, profile).
    """
    y, x = np.indices(data.shape)
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2).astype(int)
    valid = mask.astype(bool)
    counts = np.bincount(r[valid].ravel())
    sums   = np.bincount(r[valid].ravel(), data[valid].ravel())
    profile = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    radii_um = np.arange(len(profile)) * SCALE_UM_PER_PX
    return radii_um, profile


def truncate_profile(radii_um, profile):
    """
    Preserves the original truncation logic from the StarDist version:
    Beyond MAX_RADIUS_UM, keep only values that are <= the minimum in that region;
    anything larger becomes NaN. NaNs are later interpolated across for plotting.
    """
    profile = profile.copy()
    post_mask = radii_um > MAX_RADIUS_UM
    if post_mask.any():
        post_vals = profile[post_mask]
        if np.any(np.isfinite(post_vals)):
            min_post = np.nanmin(post_vals)
            post_vals = np.where(post_vals > min_post, np.nan, post_vals)
            profile[post_mask] = post_vals
    return profile


## Per-image analysis

In [ ]:
def full_analysis(path, out_root, plots_root):
    img_name = os.path.basename(path)
    plot_dir = os.path.join(plots_root, img_name.replace('.tif', ''))
    os.makedirs(plot_dir, exist_ok=True)

    img = load_and_preprocess(path)
    if img.shape[0] != len(CHANNEL_NAMES):
        raise ValueError(
            f"Expected {len(CHANNEL_NAMES)} channels {CHANNEL_NAMES}, "
            f"got {img.shape[0]} in {img_name}"
        )

    dapi_enh = enhance_dapi(img[DAPI_IDX])
    labels, mask = cellpose_mask(dapi_enh)
    if mask.sum() == 0:
        raise RuntimeError("Cellpose found no nuclei — check DAPI channel or tune parameters")
    center = compute_center_float(mask)

    # --- Max projection panels ---
    fig, axs = plt.subplots(1, img.shape[0], figsize=(4*img.shape[0], 4))
    for i in range(img.shape[0]):
        axs[i].imshow(img[i], cmap='gray')
        axs[i].set_title(f"Max Projection: {CHANNEL_NAMES[i]}")
        axs[i].axis('off')
    save_figure(fig, os.path.join(plot_dir, 'max_projection.png'))

    # --- Segmentation overlay (with center marker) ---
    H, W = img[DAPI_IDX].shape
    extent = [-W/2*SCALE_UM_PER_PX, W/2*SCALE_UM_PER_PX,
              -H/2*SCALE_UM_PER_PX, H/2*SCALE_UM_PER_PX]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(dapi_enh, cmap='gray', extent=extent, origin='lower')
    for c in find_contours(labels, 0.5):
        ax.plot((c[:, 1] - W/2) * SCALE_UM_PER_PX,
                (c[:, 0] - H/2) * SCALE_UM_PER_PX,
                lw=1, color='lime')
    ax.plot((center[0] - W/2) * SCALE_UM_PER_PX,
            (center[1] - H/2) * SCALE_UM_PER_PX,
            'y+', markersize=20, mew=3, label='Center')
    ax.set_title(f"DAPI + Cellpose-SAM ({int(labels.max())} nuclei detected)")
    ax.legend(loc='lower right')
    ax.axis('equal')
    save_figure(fig, os.path.join(plot_dir, 'segmentation.png'))

    # --- Channel heatmaps ---
    fig, axs = plt.subplots(1, img.shape[0], figsize=(4*img.shape[0], 4))
    for i in range(img.shape[0]):
        axs[i].imshow(img[i], cmap=CHANNEL_CMAPS[i], extent=extent, origin='lower')
        axs[i].set_title(CHANNEL_NAMES[i])
        axs[i].axis('equal')
    plt.tight_layout()
    save_figure(fig, os.path.join(plot_dir, 'heatmaps.png'))

    # --- Radial profiles (matches original order: normalize → truncate → renormalize) ---
    profiles = {}
    full_radius = None
    for c in range(img.shape[0]):
        radii, profile = radial_profile_masked(img[c], center, mask)

        # First normalize
        if NORMALIZE_PROFILES:
            peak = np.nanmax(profile)
            if peak and peak > 0:
                profile = profile / peak

        # Original NaN-truncation trick past MAX_RADIUS_UM
        profile = truncate_profile(radii, profile)

        # Renormalize after truncation
        if NORMALIZE_PROFILES:
            peak = np.nanmax(profile)
            if peak and peak > 0:
                profile = profile / peak

        # CSV keeps the raw NaN version; plot uses interpolated version
        profiles[f'C{c}'] = profile
        if full_radius is None:
            full_radius = radii

        profile_plot = pd.Series(profile).interpolate(limit_direction='both')

        fig, ax = plt.subplots()
        ax.plot(radii, profile_plot, lw=2, color=CHANNEL_COLORS[c])
        ax.set_xlim(0, PLOT_MAX_UM)
        ax.set_xlabel("Distance (µm)")
        ax.set_ylabel("Normalized Intensity" if NORMALIZE_PROFILES else "Intensity")
        ax.set_title(f"{CHANNEL_NAMES[c]} Radial Profile")
        save_figure(fig, os.path.join(plot_dir, f"profile_C{c}.png"))

    # --- Save per-image CSV ---
    df = pd.DataFrame(profiles)
    df.insert(0, 'Radius (µm)', full_radius)
    csv_path = os.path.join(out_root, img_name.replace('.tif', '_radial.csv'))
    df.to_csv(csv_path, index=False)

    # --- Combined per-image plot (interpolated over NaN for plotting) ---
    fig, ax = plt.subplots(figsize=(6, 4))
    for c in range(img.shape[0]):
        series = pd.Series(profiles[f'C{c}']).interpolate(limit_direction='both')
        ax.plot(full_radius, series, lw=2,
                label=CHANNEL_NAMES[c], color=CHANNEL_COLORS[c])
    ax.set_xlim(0, PLOT_MAX_UM)
    ax.set_xlabel("Distance (µm)")
    ax.set_ylabel("Normalized Intensity" if NORMALIZE_PROFILES else "Intensity")
    ax.set_title("Combined Radial Profiles")
    ax.legend()
    save_figure(fig, os.path.join(plot_dir, 'combined_profiles.png'))

    return df.assign(Image=img_name), plot_dir


## PowerPoint slide layout

In [ ]:
def add_image_summary_slide(prs, plot_dir, image_name):
    slide = prs.slides.add_slide(prs.slide_layouts[5])
    tb = slide.shapes.add_textbox(Inches(0.3), Inches(0.05), Inches(12), Inches(0.5))
    tb.text_frame.text = f"Results for {image_name}"
    layout = {
        'combined_profiles.png': (8, 0.5),
        'profile_C0.png': (0.3, 1.2),
        'profile_C1.png': (3.0, 1.2),
        'profile_C2.png': (5.7, 1.2),
        'max_projection.png': (0.3, 3.9),
        'segmentation.png':   (3.0, 3.9),
        'heatmaps.png':       (5.7, 3.9),
    }
    for fname, (x, y) in layout.items():
        p = os.path.join(plot_dir, fname)
        if os.path.exists(p):
            slide.shapes.add_picture(p, Inches(x), Inches(y),
                                     width=Inches(2.5), height=Inches(2.5))


## Batch run

Set `BASE_DIR` below to the folder containing your `.tif` files, then run this cell.

Outputs are written under `BASE_DIR`:
- `radial_outputs/` — one CSV per image (all three channels)
- `plots/<image>/` — per-image PNG plots
- `summary/` — combined CSV, per-channel cross-image plots, and `failed_images.log`
- `ppt_reports/batch_report.pptx` — PowerPoint summary


In [ ]:
# CHANGE THIS to point at your folder of TIFF images
BASE_DIR = "PATH_TO_YOUR_TIFF_FOLDER"

if BASE_DIR == "PATH_TO_YOUR_TIFF_FOLDER":
    print("⚠ Set BASE_DIR above to your TIFF folder path, then re-run.")
else:
    out_root      = os.path.join(BASE_DIR, "radial_outputs")
    plots_root    = os.path.join(BASE_DIR, "plots")
    ppt_root      = os.path.join(BASE_DIR, "ppt_reports")
    summary_root  = os.path.join(BASE_DIR, "summary")
    for d in (out_root, plots_root, ppt_root, summary_root):
        os.makedirs(d, exist_ok=True)

    tif_files = sorted(glob.glob(os.path.join(BASE_DIR, "*.tif")))
    print(f"Found {len(tif_files)} TIFF file(s) in {BASE_DIR}")

    all_dfs = []
    batch_ppt = Presentation()
    failure_log = os.path.join(summary_root, "failed_images.log")
    open(failure_log, 'w').close()   # start with an empty log

    for fpath in tif_files:
        fname = os.path.basename(fpath)
        try:
            df, plot_dir = full_analysis(fpath, out_root, plots_root)
            all_dfs.append(df)
            add_image_summary_slide(batch_ppt, plot_dir, fname)
            print(f"  ✓ {fname}")
        except Exception as e:
            with open(failure_log, 'a') as f:
                f.write(f"{fname}\t{type(e).__name__}: {e}\n")
            print(f"  ✗ {fname} — {type(e).__name__}: {e}  (logged)")

    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True)
        combined.to_csv(os.path.join(summary_root, "combined_radial_profiles.csv"), index=False)

        for c_idx, c_name in enumerate(CHANNEL_NAMES):
            fig, ax = plt.subplots(figsize=(8, 6))
            for name, group in combined.groupby("Image"):
                series = pd.Series(group[f'C{c_idx}'].values).interpolate(limit_direction='both')
                ax.plot(group['Radius (µm)'], series,
                        alpha=0.8, label=clean_label(name))
            ax.set_title(f"Combined Radial Profiles: {c_name}")
            ax.set_xlabel("Distance from Center (µm)")
            ax.set_ylabel("Normalized Intensity" if NORMALIZE_PROFILES else "Intensity")
            ax.set_xlim(0, PLOT_MAX_UM)
            ax.legend(fontsize=7, loc="lower right", ncol=1, frameon=False)
            fig.savefig(os.path.join(summary_root, f"combined_profiles_{c_name}.png"),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)

        batch_ppt.save(os.path.join(ppt_root, "batch_report.pptx"))
        print(f"\nSaved combined CSV, per-channel plots, and PowerPoint to {summary_root}")
    else:
        print("\nNo images were successfully processed. Check the failure log.")
